<a href="https://colab.research.google.com/github/snehithreddy868/AI-Chatbot-/blob/main/ML_code_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

data = load_iris()

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(data.data, data.target)

print("Model trained successfully")

Model trained successfully


In [3]:
import os
import joblib
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

class DataPipeline:

    def __init__(self, artifact_dir="artifacts"):
        self.artifact_dir = artifact_dir
        self.scaler = StandardScaler()
        self.model = RandomForestClassifier(
            n_estimators=100, random_state=42
        )
        os.makedirs(artifact_dir, exist_ok=True)

    def extract(self):
        data = load_iris(as_frame=True)
        df = data.frame
        print(f"[EXTRACT ] loaded {df.shape[0]} rows x {df.shape[1]} cols")
        return df

    def validate(self, df):
        assert df.isnull().sum().sum() == 0
        assert "target" in df.columns
        print("[VALIDATE] no missing values, target present -> OK")
        return df

    def transform(self, df):
        X = df.drop(columns=["target"])
        y = df["target"]
        X_scaled = self.scaler.fit_transform(X)
        print(f"[TRANSFORM] standard-scaled {X.shape[1]} features")
        return X_scaled, y

    def train(self, X, y):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        self.model.fit(X_train, y_train)
        y_pred = self.model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        print(f"[TRAIN ] model fitted, test accuracy = {accuracy:.3f}")
        print(classification_report(y_test, y_pred, digits=2))

    def persist(self):
        joblib.dump(self.model, "artifacts/model.pkl")
        joblib.dump(self.scaler, "artifacts/scaler.pkl")
        print("[PERSIST ] saved model -> artifacts/model.pkl")
        print("[PERSIST ] saved scaler -> artifacts/scaler.pkl")

    def run(self):
        print("=== Automated Data Pipeline Started ===")
        df = self.extract()
        df = self.validate(df)
        X, y = self.transform(df)
        self.train(X, y)
        self.persist()
        print("=== Pipeline Completed Successfully ===")

if __name__ == "__main__":
    DataPipeline().run()

=== Automated Data Pipeline Started ===
[EXTRACT ] loaded 150 rows x 5 cols
[VALIDATE] no missing values, target present -> OK
[TRANSFORM] standard-scaled 4 features
[TRAIN ] model fitted, test accuracy = 0.933
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

[PERSIST ] saved model -> artifacts/model.pkl
[PERSIST ] saved scaler -> artifacts/scaler.pkl
=== Pipeline Completed Successfully ===
